In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Repeat the cleaning from Phase 2
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df.drop(columns=['customerID'], inplace=True)

In [2]:
# ── 1. Tenure-based features ─────────────────────────────────
# EDA showed low tenure → high churn. Bucket it for non-linear patterns.
df['tenure_group'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['0-1yr', '1-2yr', '2-4yr', '4-6yr']
)

# Flag for "new customer" — highest risk segment from EDA
df['is_new_customer'] = (df['tenure'] <= 12).astype(int)

# ── 2. Service adoption count ────────────────────────────────
# Customers using more add-on services tend to be more "locked in"
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                 'TechSupport', 'StreamingTV', 'StreamingMovies']

df['total_services'] = (df[service_cols] == 'Yes').sum(axis=1)

# ── 3. Average monthly spend ratio ───────────────────────────
# Catches inconsistencies — e.g. new customers with disproportionately high charges
df['avg_monthly_spend'] = df['TotalCharges'] / (df['tenure'] + 1)  # +1 avoids div/0

# ── 4. Contract risk flag ────────────────────────────────────
# EDA: month-to-month = 43% churn vs ~3% for long contracts
df['is_month_to_month'] = (df['Contract'] == 'Month-to-month').astype(int)

# ── 5. High-risk payment method flag ─────────────────────────
df['is_electronic_check'] = (df['PaymentMethod'] == 'Electronic check').astype(int)

# ── 6. No internet service flag (impacts multiple columns) ───
df['has_internet'] = (df['InternetService'] != 'No').astype(int)

print(df[['tenure_group', 'is_new_customer', 'total_services',
          'avg_monthly_spend', 'is_month_to_month']].head())

  tenure_group  is_new_customer  total_services  avg_monthly_spend  \
0        0-1yr                1               1          14.925000   
1        2-4yr                0               2          53.985714   
2        0-1yr                1               2          36.050000   
3        2-4yr                0               3          40.016304   
4        0-1yr                1               0          50.550000   

   is_month_to_month  
0                  1  
1                  0  
2                  1  
3                  0  
4                  1  


In [3]:
for col in service_cols + ['MultipleLines']:
    df[col] = df[col].replace('No internet service', 'No')
    df[col] = df[col].replace('No phone service', 'No')

In [4]:
from sklearn.preprocessing import LabelEncoder

# Binary categoricals (Yes/No, Male/Female) → Label encoding (0/1)
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
                'PaperlessBilling', 'MultipleLines'] + service_cols

le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

# Multi-class categoricals → One-hot encoding (no ordinal relationship)
multiclass_cols = ['InternetService', 'Contract', 'PaymentMethod', 'tenure_group']

df = pd.get_dummies(df, columns=multiclass_cols, drop_first=True)

In [5]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train shape: {X_train.shape}, churn rate: {y_train.mean():.1%}")
print(f"Test shape: {X_test.shape}, churn rate: {y_test.mean():.1%}")

Train shape: (5634, 32), churn rate: 26.5%
Test shape: (1409, 32), churn rate: 26.5%


In [6]:
from imblearn.over_sampling import SMOTE

print(f"Before SMOTE: {y_train.value_counts().to_dict()}")

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"After SMOTE: {y_train_resampled.value_counts().to_dict()}")

Before SMOTE: {0: 4139, 1: 1495}
After SMOTE: {0: 4139, 1: 4139}


In [7]:
import os
os.makedirs('../data/processed', exist_ok=True)

X_train_resampled.to_csv('../data/processed/X_train.csv', index=False)
y_train_resampled.to_csv('../data/processed/y_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Also save the UNBALANCED train set for comparison experiments
X_train.to_csv('../data/processed/X_train_original.csv', index=False)
y_train.to_csv('../data/processed/y_train_original.csv', index=False)